In [2]:
import joblib
import cv2
import pandas as pd
from tensorflow.keras.models import load_model

# Test pour le SVC

In [3]:

def detect_digits(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edged = cv2.Canny(blurred, 30, 150)
    contours, _ = cv2.findContours(edged.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    digits_rois = []
    for c in contours:
        (x, y, w, h) = cv2.boundingRect(c)
        if w >= 5 and h >= 25:
            roi = gray[y:y + h, x:x + w]
            digits_rois.append((x, y, w, h, roi))
    return img, digits_rois

def square(img, digits_roi):
    for (x, y, w, h, _) in digits_roi:
        cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 2)
    return img
    

def recognize_digits(img, digits_rois, model):
    digits = []
    for (x, y, w, h, roi) in digits_rois:
        roi = cv2.resize(roi, (28, 28), interpolation=cv2.INTER_AREA)
        roi = roi / 255.0  # Normalisation simple
        roi = roi.reshape(1, 28*28)  # Aplatir l'image pour correspondre à l'entrée du modèle

        # Utiliser les noms de caractéristiques corrects
        feature_names = [f'pixel{i}' for i in range(1, 28*28 + 1)]
        roi_df = pd.DataFrame(roi, columns=feature_names)

        digit = model.predict(roi_df)
        probabilities = model.predict_proba(roi_df)[0]
        print(f"Digit: {digit[0]}, Probabilities: {probabilities}")

        digit = digit[0]  # Obtenir la prédiction réelle à partir du résultat
        digits.append((x, y, digit, probabilities))
        cv2.putText(img, str(digit), (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 2)
    cv2.imshow("Digits Recognized", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    return digits


In [4]:
image_path = './data/1_.png'
img, digits_rois = detect_digits(image_path)
img = square(img, digits_rois)
print(digits_rois)

[(464, 183, 132, 631, array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=uint8))]


In [5]:
model = joblib.load('./models/DetectionReconize_optimized2.pkl')
digits = recognize_digits(img, digits_rois, model)
print(digits)

c:\Python311\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
c:\Python311\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


Digit: 2, Probabilities: [0.04806745 0.00747742 0.55393679 0.08907809 0.01353417 0.11286766
 0.0464132  0.03838427 0.06370973 0.02653122]
[(464, 183, '2', array([0.04806745, 0.00747742, 0.55393679, 0.08907809, 0.01353417,
       0.11286766, 0.0464132 , 0.03838427, 0.06370973, 0.02653122]))]


# Test pour le CNN

In [6]:
def recognize_digits_cnn(img, digits_rois, model):
    digits = []
    for (x, y, w, h, roi) in digits_rois:
        roi = cv2.resize(roi, (28, 28), interpolation=cv2.INTER_AREA)
        roi = roi / 255.0  # Normalisation simple
        roi = roi.reshape(1, 28, 28, 1)  # Reshape pour correspondre à l'entrée du modèle CNN (batch_size, height, width, channels)

        digit = model.predict(roi)
        digit_class = digit.argmax()  # Obtenir la prédiction réelle à partir du résultat
        probabilities = digit[0]  # Obtenir les probabilités

        # Afficher les résultats sans encodage
        #?print(f"Digit: {digit_class}, Probabilities: {probabilities}")

        digits.append((x, y, digit_class, probabilities))
        cv2.putText(img, str(digit_class), (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 2)
    cv2.imshow("Digits Recognized", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    return digits

In [7]:
model_CNN = load_model('./models/DetectionReconize_CNN.h5')  # Utiliser load_model pour le modèle CNN
digits_Cnn = recognize_digits_cnn(img, digits_rois, model_CNN)

print(digits_Cnn)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
[(464, 183, 0, array([0.8563115 , 0.05638446, 0.00266583, 0.00135217, 0.01045401,
       0.00340797, 0.03762892, 0.00696067, 0.01711254, 0.00772192],
      dtype=float32))]
